# Use Qdrant In NAIVE RAG 

In [32]:
# Import dataset
import warnings
warnings.filterwarnings('ignore')
from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader('Why_Language_Models_Hallucinate_Explainer.pdf') 
pages = loader.load()

In [33]:
# Split Dataset 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1200 , chunk_overlap=150)
texts = text_spliter.split_documents(pages)
chunks = [i.page_content for i in texts]
metadata = [i.metadata for i in texts]
metadata[0]

{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-07-02T09:06:07+00:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2026-07-02T09:06:07+00:00',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'trapped': '/False',
 'source': 'Why_Language_Models_Hallucinate_Explainer.pdf',
 'total_pages': 3,
 'page': 0,
 'page_label': '1'}

In [34]:
# Embedding Creates 
from sentence_transformers import SentenceTransformer 
embed_transformer = SentenceTransformer(model_name_or_path='all-MiniLM-L6-v2',similarity_fn_name="cosine") 
embeddings = embed_transformer.encode(chunks)
embeddings[0]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6747.15it/s]


array([-5.90949832e-03,  9.30776633e-03,  2.32948121e-02,  4.51134518e-02,
        8.67218338e-03,  1.04130358e-02, -7.25856647e-02,  3.60563919e-02,
        4.88252118e-02, -1.94204468e-02, -8.73478055e-02, -6.71620155e-03,
        9.39331427e-02, -3.46006220e-03, -4.78915498e-03, -4.92812134e-02,
        3.80466948e-03,  7.45089278e-02, -4.62084934e-02, -1.18435867e-01,
        3.01859546e-02,  6.58174455e-02,  2.83327308e-02,  3.20922546e-02,
        6.63133487e-02, -4.73553762e-02, -6.58300295e-02, -6.53699785e-02,
       -3.01430132e-02, -3.97976413e-02, -2.03339346e-02,  1.41706541e-01,
       -2.04547625e-02,  4.15820740e-02, -1.41619639e-02,  5.55828996e-02,
       -6.57157153e-02,  3.20819281e-02,  6.99062571e-02, -4.40365113e-02,
       -4.21014279e-02, -3.46233845e-02,  6.82488084e-02,  1.82719678e-02,
        8.51330608e-02, -6.13539815e-02, -4.61315736e-02,  1.14408173e-02,
       -1.33124813e-01, -3.89245972e-02, -1.06519647e-01, -2.60350183e-02,
        5.61646596e-02,  

In [ ]:
# Cell 1 — imports + client 
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

collection_name = "Model_Halucination"

try:
    client.collection_exists(collection_name)
except (NameError, RuntimeError):
    client = QdrantClient(path="./Naive_RAG")

In [23]:
# Cell 2 — create collection
if not client.collection_exists(collection_name=collection_name):
    client.create_collection(
        collection_name=collection_name, vectors_config=VectorParams(size=384, distance=Distance.EUCLID)
    )

In [ ]:
points = []
for i, (chunk, meta) in enumerate(zip(chunks, metadata)):
    vector = embed_transformer.encode(chunk).tolist()
    payload = meta
    payload["text"] = chunk
    points.append(PointStruct(id=i, vector=vector, payload=payload))

client.upsert(collection_name=collection_name, points=points)
print(f'the length of points {len(points)}')

the length of points 10


In [28]:
def retrive_chunks(query: str, threshold: float = 1.4, k_top: int = 3):
    query_enccode = embed_transformer.encode(query).tolist()
    result = client.query_points(collection_name=collection_name, query=query_enccode, limit=k_top).points
    near_chunks = [r.payload["text"] for r in result if r.score < threshold]
    return "\n\n".join(near_chunks) if near_chunks else "Not Relevent Content"

retrive_chunks("what is the full form of LLM ?", threshold=1.4)

'Generative Large Language Models. EMNLP, 2023.\n[4] Ji, Z. et al. Survey of Hallucination in Natural Language Generation. ACM Computing Surveys, 2023.\n \n[5] Survey and Analysis of Hallucinations in Large Language Models: Attribution to Prompting Strategies or Model\n Behavior. Frontiers in Artificial Intelligence, 2025.\n[6] Hallucination to Truth: A Review of Fact-Checking and Factuality Evaluation in Large Language Models.\n arXiv:2508.03860, 2025.\n[7] Ansari, S. Compound Deception in Elite Peer Review: A Failure Mode Taxonomy of 100 Fabricated Citations at\n NeurIPS 2025. arXiv:2602.05930, 2026.\n\ngeneration for high-stakes domains such as medicine and law.\n7. Takeaways\n- Hallucination is a predictable statistical outcome of current training and evaluation norms, not an\n unexplained defect.\n- Some hallucination is mathematically unavoidable for facts seen only once in training, regardless of\n model scale.\n- Post-training reduces certain error classes but cannot fully remo

In [31]:
q1 ="what is the full form of LLM"
content = retrive_chunks(query=q1)

prompt = f"""
Provide users questions answers based on the local provided data : 
content : {content} 
qustion : {q1}
"""

import os 
from dotenv import load_dotenv 
from langchain_groq  import ChatGroq 
groq = ChatGroq(model='llama-3.1-8b-instant')
groq.invoke(prompt).content

'The full form of LLM is Large Language Model'